In [1]:
from __future__ import annotations

import random
from pathlib import Path
import csv
import json
from typing import List, Tuple, Optional, Dict, Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
try:
    from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import LabelEncoder, label_binarize
except Exception as e:
    raise ImportError(
        "scikit-learn import failed in this Jupyter kernel. Install/repair scikit-learn, then restart kernel."
    ) from e

from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# ====================== CONSTANTS ======================
SPLIT_SEED = 42                     # fixed split seed (never changes)
INIT_SEEDS = list(range(30))        # 30 initialization seeds
RESULTS_DIR = Path("/mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/Results/AL_MODELS")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_CSV = RESULTS_DIR / "experiment_summary_subject_fulltest.csv"
DETAILS_CSV = RESULTS_DIR / "subject_fulltest_subject_level_details.csv"
DETAILS_META_JSON = RESULTS_DIR / "subject_fulltest_subject_level_meta.jsonl"

EMB_ROOT = Path("/mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/embeddings")
HUBERT_DIR = EMB_ROOT / "Hubert"
DATA2VEC_DIR = EMB_ROOT / "data2vec"
WAV2VEC_DIR = EMB_ROOT / "wav2vec"
WAVLM_DIR = EMB_ROOT / "wavlm"

# Hyperparameters (fixed for all runs)
N_WAY = None           # will be set after loading data
K_SHOT = 5
Q_QUERY = 15

# When running ALL configs, skip these config names
SKIP_CONFIGS_WHEN_ALL = {"all"}


MODEL_CFG = dict(
    hidden_dim=512,
    feature_dim=256,
    context_dim=64,
    use_class_lstm=True,
    use_feature_net=True,
    prototype_agg="lstm",
    feature_net_n_hidden=2,
    dropout_p1=0.3,
    dropout_p2=0.2,
    use_metric_learner=True,
    distance_metric="cosine",
    distance_scale_init=10.0,
)

TRAIN_CFG = dict(
    num_epochs=120,
    meta_batch_size=16,
    patience=10**9,
    lr=3e-5,
    weight_decay=0.0,
    scheduler_patience=10,
    scheduler_factor=0.5,
    explicit_l2=True,
)

EVAL_CFG = dict(
    test_task_multiplier=2,
)

# Debug prints for SUBJECT-wise evaluation (Notebook 3 additions only)
DEBUG_SUBJECT = True
DEBUG_SUBJECT_MAX = 10   # print first N subjects
DEBUG_SUBJECT_ONLY_FIRST_SEED = True  # avoid spam across 30 seeds


# ====================== HELPER FUNCTIONS ======================
def ask_yes_no(prompt: str, default: str = "n") -> bool:
    raw = input(f"{prompt} [y/n, default={default}]: ").strip().lower()
    if raw == "":
        raw = default
    if raw not in {"y", "n"}:
        raise ValueError("Please enter 'y' or 'n'")
    return raw == "y"

def ask_index(prompt: str, *, default: int, max_index: int, allow_all: bool = False) -> int:
    extra = " (or -1 for ALL)" if allow_all else ""
    raw = input(f"{prompt} [default={default}]{extra}: ").strip()
    if raw == "":
        return default
    idx = int(raw)
    if allow_all and idx == -1:
        return -1
    if not (0 <= idx <= max_index):
        raise ValueError(f"Index out of range: {idx} (valid 0..{max_index})")
    return idx

def discover_models(emb_root: Path) -> List[str]:
    preferred_order = ["Hubert", "data2vec", "wav2vec", "wavlm"]
    found: List[str] = []
    for name in preferred_order:
        p = emb_root / name
        if not p.is_dir():
            continue
        if not (p / "labels.npy").exists():
            continue
        if not (p / "subject_ids.npy").exists():
            continue
        has_x = any(f.is_file() and f.suffix == ".npy" and f.name.startswith("x") for f in p.iterdir())
        if not has_x:
            continue
        found.append(name)
    return found

def _cfg_name_from_file(model_name: str, f: Path) -> str:
    stem = f.stem
    if model_name == "Hubert" and stem.startswith("xlarge_hubert_"):
        return stem[len("xlarge_hubert_"):]
    if model_name == "data2vec" and stem.startswith("xdata2vec_"):
        return stem[len("xdata2vec_"):]
    if model_name == "wav2vec" and stem.startswith("xwav2vec2_"):
        return stem[len("xwav2vec2_"):]
    if model_name == "wavlm" and stem.startswith("xwavlm_"):
        return stem[len("xwavlm_"):]
    stem2 = stem[1:] if stem.startswith("x") else stem
    return stem2.split("_", 1)[1] if "_" in stem2 else stem2

def discover_configs(model_dir: Path, model_name: str) -> List[Tuple[str, Path]]:
    configs: List[Tuple[str, Path]] = []
    for f in sorted(model_dir.iterdir()):
        if not (f.is_file() and f.suffix == ".npy" and f.name.startswith("x")):
            continue
        configs.append((_cfg_name_from_file(model_name, f), f))
    preferred = ["acoustic", "all", "fullspec", "last3", "middle3", "stability"]
    order = {name: i for i, name in enumerate(preferred)}
    configs.sort(key=lambda x: (order.get(x[0], 999), x[0]))
    return configs

def load_data(model_name: str, config_path: Path, model_dir: Path):
    """Load X, y, subject_ids, encode labels, and perform fixed subject-wise split."""
    X = np.load(config_path)
    y_raw = np.load(model_dir / "labels.npy", allow_pickle=True)
    subject_ids = np.load(model_dir / "subject_ids.npy", allow_pickle=True).astype(str)

    assert len(X) == len(y_raw) == len(subject_ids), "Length mismatch"

    le = LabelEncoder()
    y = le.fit_transform(y_raw)
    n_classes = len(le.classes_)

    # Subject-level majority label
    uniq_subjects = np.unique(subject_ids)
    subj_y = np.zeros(len(uniq_subjects), dtype=int)
    for i, sid in enumerate(uniq_subjects):
        ys = y[subject_ids == sid]
        vals, cnts = np.unique(ys, return_counts=True)
        subj_y[i] = int(vals[np.argmax(cnts)])

    # Subject-wise stratified split (80/10/10) with FIXED split seed
    subj_train, subj_temp, ysubj_train, ysubj_temp = train_test_split(
        uniq_subjects, subj_y, test_size=0.2, stratify=subj_y, random_state=SPLIT_SEED
    )
    subj_val, subj_test, ysubj_val, ysubj_test = train_test_split(
        subj_temp, ysubj_temp, test_size=0.5, stratify=ysubj_temp, random_state=SPLIT_SEED
    )

    set_train = set(subj_train)
    set_val = set(subj_val)
    set_test = set(subj_test)

    train_idx = np.where(np.isin(subject_ids, list(set_train)))[0]
    val_idx = np.where(np.isin(subject_ids, list(set_val)))[0]
    test_idx = np.where(np.isin(subject_ids, list(set_test)))[0]

    X_train, y_train = X[train_idx], y[train_idx]
    X_val, y_val = X[val_idx], y[val_idx]
    X_test, y_test = X[test_idx], y[test_idx]

    subject_ids_val = subject_ids[val_idx]
    subject_ids_test = subject_ids[test_idx]

    return X_train, y_train, X_val, y_val, X_test, y_test, subject_ids_val, subject_ids_test, le, n_classes

def maybe_apply_smote(X_train, y_train, do_smote, seed=SPLIT_SEED):
    if do_smote:
        try:
            from imblearn.over_sampling import SMOTE
        except ImportError:
            raise ImportError("imbalanced-learn required for SMOTE")
        sm = SMOTE(random_state=seed)
        X_train, y_train = sm.fit_resample(X_train, y_train)
    return X_train, y_train

def normalize(X_train, X_val, X_test):
    train_mean = X_train.mean(axis=0)
    train_std = X_train.std(axis=0) + 1e-8
    X_train = (X_train - train_mean) / train_std
    X_val = (X_val - train_mean) / train_std
    X_test = (X_test - train_mean) / train_std
    return X_train, X_val, X_test

# ====================== TASK GENERATOR ======================
class BalancedTaskGenerator:
    def __init__(self, features, labels, n_way, k_shot, q_query):
        self.features = features
        self.labels = labels
        self.n_way = n_way
        self.k_shot = k_shot
        self.q_query = q_query

        self.class_indices = {}
        for idx, label in enumerate(labels):
            if label not in self.class_indices:
                self.class_indices[label] = []
            self.class_indices[label].append(idx)

        self.classes = list(self.class_indices.keys())

    def create_task(self):
        selected_classes = random.sample(self.classes, self.n_way)

        support_set = []
        query_set = []

        for cls in selected_classes:
            samples = random.sample(self.class_indices[cls], self.k_shot + self.q_query)
            support_set.extend([(samples[i], cls) for i in range(self.k_shot)])
            query_set.extend([(samples[i], cls) for i in range(self.k_shot, self.k_shot + self.q_query)])

        support_features = torch.stack([torch.FloatTensor(self.features[idx]) for idx, _ in support_set])
        support_labels = torch.LongTensor([label for _, label in support_set])

        query_features = torch.stack([torch.FloatTensor(self.features[idx]) for idx, _ in query_set])
        query_labels = torch.LongTensor([label for _, label in query_set])

        return support_features, support_labels, query_features, query_labels

# ====================== MODEL DEFINITIONS ======================
class ClassLSTM(nn.Module):
    def __init__(self, feature_dim, context_dim):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=feature_dim,
            hidden_size=context_dim,
            num_layers=1,
            batch_first=True,
        )
        self.fc = nn.Sequential(
            nn.Linear(context_dim, feature_dim),
            nn.Tanh(),
        )

    def forward(self, x):
        output, _ = self.lstm(x)
        h_mean = output.mean(dim=1)
        return self.fc(h_mean)

class ClassCNN(nn.Module):
    """Lightweight prototype aggregator to replace ClassLSTM.

    Input: (1, n_support, feature_dim)
    Output: (feature_dim,)
    """

    def __init__(self, feature_dim: int, context_dim: int):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(feature_dim, context_dim, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.fc = nn.Sequential(
            nn.Linear(context_dim, feature_dim),
            nn.Tanh(),
        )

    def forward(self, x):
        # x: (1, T, F) -> (1, F, T)
        x = x.transpose(1, 2)
        h = self.conv(x).squeeze(-1)  # (1, context_dim)
        return self.fc(h.squeeze(0))


class HyperMetaLearner(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim,
        feature_dim,
        context_dim,
        num_classes,
        use_metric_learner=False,
        distance_metric="euclidean",
        distance_scale_init=10.0,
        dropout_p1=0.3,
        dropout_p2=0.2,
        *,
        use_class_lstm: bool = True,
        use_feature_net: bool = True,
        prototype_agg: str | None = None,  # None -> use_class_lstm decides
        feature_net_n_hidden: int = 2,
    ):
        super().__init__()

        self.num_classes = int(num_classes)
        self.use_metric_learner = bool(use_metric_learner)
        self.distance_metric = str(distance_metric)
        self.use_class_lstm = bool(use_class_lstm)
        self.use_feature_net = bool(use_feature_net)

        # Decide prototype aggregator
        if prototype_agg is None:
            self.prototype_agg = "lstm" if self.use_class_lstm else "mean"
        else:
            self.prototype_agg = str(prototype_agg).lower()

        if self.prototype_agg not in {"lstm", "cnn", "mean"}:
            raise ValueError(f"Unsupported prototype_agg: {self.prototype_agg}")

        if self.use_feature_net:
            self.feature_dim = int(feature_dim)

            n_hidden = int(feature_net_n_hidden)
            if n_hidden < 1:
                raise ValueError(f"feature_net_n_hidden must be >= 1, got {n_hidden}")

            layers: list[nn.Module] = []

            # First hidden block
            layers += [
                nn.Linear(input_dim, hidden_dim),
                nn.LayerNorm(hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout_p1),
            ]

            # Additional hidden blocks (hidden_dim -> hidden_dim)
            for _ in range(n_hidden - 1):
                layers += [
                    nn.Linear(hidden_dim, hidden_dim),
                    nn.LayerNorm(hidden_dim),
                    nn.GELU(),
                    nn.Dropout(dropout_p2),
                ]

            # Output projection
            layers += [
                nn.Linear(hidden_dim, feature_dim),
                nn.LayerNorm(feature_dim),
                nn.Tanh(),
            ]

            self.feature_net = nn.Sequential(*layers)
        else:
            # No adapter: operate directly in the original embedding space.
            self.feature_dim = int(input_dim)
            self.feature_net = nn.Identity()

        self.class_lstm = ClassLSTM(feature_dim=self.feature_dim, context_dim=context_dim)
        self.class_cnn = ClassCNN(feature_dim=self.feature_dim, context_dim=context_dim)

        self.metric_learner = (
            nn.Sequential(
                nn.Linear(self.feature_dim, self.feature_dim),
                nn.ReLU(),
                nn.Linear(self.feature_dim, self.feature_dim),
            )
            if self.use_metric_learner
            else nn.Identity()
        )

        self.distance_scale = nn.Parameter(torch.tensor(float(distance_scale_init)))

    def forward(self, x):
        return self.feature_net(x)

    def compute_prototypes(self, support, support_labels, n_way):
        device_ = next(self.parameters()).device
        support_features = self.feature_net(support)
        prototypes = torch.zeros(int(n_way), int(self.feature_dim), device=device_)

        for cls in range(int(n_way)):
            class_mask = support_labels == cls
            class_features = support_features[class_mask]
            if len(class_features) > 0:
                if self.prototype_agg == "mean":
                    prototypes[cls] = class_features.mean(dim=0)
                elif self.prototype_agg == "cnn":
                    prototypes[cls] = self.class_cnn(class_features.unsqueeze(0))
                else:  # lstm
                    prototypes[cls] = self.class_lstm(class_features.unsqueeze(0)).squeeze(0)

        return prototypes

    def compute_distances(self, prototypes, query_features):
        prototypes = prototypes + self.metric_learner(prototypes)
        query_features = query_features + self.metric_learner(query_features)

        if self.distance_metric == "euclidean":
            return torch.cdist(query_features, prototypes, p=2) * self.distance_scale
        elif self.distance_metric == "manhattan":
            return torch.cdist(query_features, prototypes, p=1) * self.distance_scale
        elif self.distance_metric == "sqeuclidean":
            q = query_features.unsqueeze(1)
            p = prototypes.unsqueeze(0)
            return ((q - p) ** 2).sum(dim=-1) * self.distance_scale
        elif self.distance_metric == "cosine":
            q = F.normalize(query_features, p=2, dim=-1)
            p = F.normalize(prototypes, p=2, dim=-1)
            return (1 - torch.matmul(q, p.T)) * self.distance_scale
        else:
            raise ValueError(f"Unsupported distance metric: {self.distance_metric}")

# ====================== LOSS & METRICS ======================
def prototypical_loss(model, support, support_labels, query, query_labels, n_way):
    device_ = next(model.parameters()).device
    support, query = support.to(device_), query.to(device_)
    support_labels, query_labels = support_labels.to(device_), query_labels.to(device_)

    query_features = model(query)

    prototypes = model.compute_prototypes(support, support_labels, n_way)
    distances = model.compute_distances(prototypes, query_features)

    log_p_y = F.log_softmax(-distances, dim=1)
    loss = F.nll_loss(log_p_y, query_labels)
    _, predictions = torch.max(log_p_y, 1)
    acc = torch.mean((predictions == query_labels).float())

    class_probs = F.softmax(-distances, dim=1).detach().cpu().numpy()

    return loss, acc, predictions, query_labels, class_probs

def calculate_auc(all_probs, all_labels, n_classes):
    if n_classes == 2:
        if all_probs.ndim == 2 and all_probs.shape[1] == 2:
            all_probs = all_probs[:, 1]
        elif all_probs.ndim == 2 and all_probs.shape[1] == 1:
            all_probs = all_probs.ravel()
        auc = roc_auc_score(all_labels, all_probs)
        return auc, [auc]
    else:
        binarized_labels = label_binarize(all_labels, classes=range(n_classes))
        auc_scores = []
        for i in range(n_classes):
            auc = roc_auc_score(binarized_labels[:, i], all_probs[:, i])
            auc_scores.append(auc)
        macro_auc = float(np.mean(auc_scores))
        return macro_auc, auc_scores



def compute_class_prototypes_from_train(model, X_tr, y_tr, n_classes, batch_size=512):
    """Compute one prototype per class using all TRAIN utterances (no episodic sampling)."""
    device_ = next(model.parameters()).device
    model.eval()
    sums = None
    counts = torch.zeros(n_classes, device=device_, dtype=torch.long)
    with torch.no_grad():
        for i in range(0, len(X_tr), batch_size):
            xb = torch.FloatTensor(X_tr[i : i + batch_size]).to(device_)
            yb = torch.LongTensor(y_tr[i : i + batch_size]).to(device_)
            feats = model(xb)
            if sums is None:
                sums = torch.zeros((n_classes, feats.shape[1]), device=device_, dtype=feats.dtype)
            for c in range(n_classes):
                m = (yb == c)
                if m.any():
                    sums[c] += feats[m].sum(dim=0)
                    counts[c] += int(m.sum().item())

    if sums is None:
        raise ValueError('No features produced while building prototypes')
    if (counts == 0).any():
        missing = [int(i) for i in (counts == 0).nonzero(as_tuple=False).view(-1).tolist()]
        raise ValueError(f'Missing class(es) in train while building prototypes: {missing}')

    protos = sums / counts.clamp(min=1).unsqueeze(1).to(sums.dtype)
    return protos


def predict_probs_from_prototypes(model, prototypes, X, batch_size=512):
    """Predict probabilities for each utterance in X using fixed class prototypes."""
    device_ = next(model.parameters()).device
    model.eval()
    all_probs = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            xb = torch.FloatTensor(X[i : i + batch_size]).to(device_)
            feats = model(xb)
            dist = model.compute_distances(prototypes, feats)
            probs = F.softmax(-dist, dim=1)
            all_probs.append(probs.detach().cpu().numpy())
    all_probs = np.vstack(all_probs)
    return all_probs


def subject_metrics_from_all_test_utterances(subject_ids, y_true, y_prob, n_classes, return_details: bool = False):
    """One prediction per subject by averaging utterance probabilities."""
    sums = {}
    counts = {}
    true_by_subj = {}
    for sid, yt, prob in zip(subject_ids, y_true, y_prob):
        sid = str(sid)
        if sid not in sums:
            sums[sid] = np.zeros((n_classes,), dtype=np.float64)
            counts[sid] = 0
            true_by_subj[sid] = int(yt)
        sums[sid] += np.asarray(prob, dtype=np.float64)
        counts[sid] += 1

    subjects = sorted(sums.keys())
    y_true_sub = np.array([true_by_subj[s] for s in subjects], dtype=int)
    y_prob_sub = np.vstack([sums[s] / max(1, counts[s]) for s in subjects])
    y_pred_sub = np.argmax(y_prob_sub, axis=1)

    correct = int(np.sum(y_pred_sub == y_true_sub))
    n_subjects = int(len(subjects))

    subj_acc = float(correct / max(1, n_subjects))
    subj_f1 = float(f1_score(y_true_sub, y_pred_sub, average='macro'))
    try:
        subj_auc = float(roc_auc_score(y_true_sub, y_prob_sub, multi_class='ovr', average='macro'))
    except ValueError:
        subj_auc = float('nan')

    if return_details:
        details = {
            'subjects': subjects,
            'n_utts': [int(counts[s]) for s in subjects],
            'y_true_sub': y_true_sub,
            'y_pred_sub': y_pred_sub,
            'y_prob_sub': y_prob_sub,
            'correct': correct,
            'n_subjects': n_subjects,
        }
        return subj_acc, subj_f1, subj_auc, details

    return subj_acc, subj_f1, subj_auc

# ====================== TRAINING & TESTING ======================
def train_model(
    model,
    train_task_gen,
    val_task_gen,          # None when merged
    n_classes,
    num_epochs=200,
    meta_batch_size=16,
    patience=20,
    lr=6e-5,
    weight_decay=1e-4,
    scheduler_patience=10,
    scheduler_factor=0.5,
    explicit_l2=True,
):
    device_ = next(model.parameters()).device
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, "max", patience=scheduler_patience, factor=scheduler_factor
    )

    best_monitor = -1e9
    best_state_dict = None
    patience_counter = 0

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        epoch_acc = 0.0

        for _ in range(meta_batch_size):
            support, support_labels, query, query_labels = train_task_gen.create_task()

            support = support.to(device_)
            query = query.to(device_)
            support_labels = support_labels.to(device_)
            query_labels = query_labels.to(device_)

            optimizer.zero_grad()
            loss, acc, _, _, _ = prototypical_loss(
                model, support, support_labels, query, query_labels, n_way=n_classes
            )

            if explicit_l2 and weight_decay != 0.0:
                l2_reg = torch.tensor(0.0, device=device_)
                for param in model.parameters():
                    l2_reg += torch.norm(param, p=2)
                loss = loss + weight_decay * l2_reg

            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            epoch_acc += acc.item()

        avg_train_loss = epoch_loss / meta_batch_size
        avg_train_acc = epoch_acc / meta_batch_size

        if val_task_gen is None:
            monitor = avg_train_acc
            scheduler.step(monitor)
        else:
            model.eval()
            val_loss = 0.0
            val_acc = 0.0
            with torch.no_grad():
                for _ in range(meta_batch_size):
                    support, support_labels, query, query_labels = val_task_gen.create_task()
                    support = support.to(device_)
                    query = query.to(device_)
                    support_labels = support_labels.to(device_)
                    query_labels = query_labels.to(device_)
                    loss, acc, _, _, _ = prototypical_loss(
                        model, support, support_labels, query, query_labels, n_way=n_classes
                    )
                    val_loss += loss.item()
                    val_acc += acc.item()
            avg_val_loss = val_loss / meta_batch_size
            avg_val_acc = val_acc / meta_batch_size
            monitor = avg_val_acc
            scheduler.step(monitor)

        if monitor > best_monitor:
            best_monitor = monitor
            best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    return best_state_dict, best_monitor

def test_model(model, test_task_gen, n_classes, meta_batch_size=16):
    device_ = next(model.parameters()).device
    model.eval()
    test_acc = 0.0
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for _ in range(meta_batch_size * 2):
            support, support_labels, query, query_labels = test_task_gen.create_task()
            support = support.to(device_)
            query = query.to(device_)
            support_labels = support_labels.to(device_)
            query_labels = query_labels.to(device_)

            _, acc, preds, true_labels, probs = prototypical_loss(
                model, support, support_labels, query, query_labels, n_way=n_classes
            )
            test_acc += acc.item()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(true_labels.cpu().numpy())
            all_probs.extend(probs)

    test_acc = test_acc / (meta_batch_size * 2)
    return test_acc, all_preds, all_labels, all_probs

# ====================== SEED LOOP FOR ONE CONFIG ======================
def run_single_seed(
    seed: int,
    X_train, y_train,
    X_val, y_val,
    X_test, y_test,
    subject_ids_val, subject_ids_test,
    n_classes: int,
    le: LabelEncoder,
    do_smote: bool,
    merge_train_val: bool,
    model_cfg: dict,
    train_cfg: dict,
    model_name: str = "",
    config_name: str = "",
) -> Dict[str, float]:
    """Run one initialization seed, return test metrics."""
    # Set all random seeds for this initialization
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Possibly apply SMOTE (using fixed SPLIT_SEED for reproducibility)
    X_tr, y_tr = maybe_apply_smote(X_train, y_train, do_smote, seed=SPLIT_SEED)

    # Normalize using training stats
    X_tr, X_va, X_te = normalize(X_tr, X_val.copy() if X_val is not None else None, X_test)

    # Subject ids aligned with X_test (and X_val if later merged)
    subj_te = np.asarray(subject_ids_test, dtype=object)

    # Merge if requested
    if merge_train_val and X_va is not None:
        X_te = np.concatenate([X_te, X_va], axis=0)
        y_test = np.concatenate([y_test, y_val], axis=0)
        subj_te = np.concatenate([subj_te, np.asarray(subject_ids_val, dtype=object)], axis=0)
        X_va = None   # signal no validation

    # Create task generators
    train_task_gen = BalancedTaskGenerator(X_tr, y_tr, n_way=n_classes, k_shot=K_SHOT, q_query=Q_QUERY)
    test_task_gen = BalancedTaskGenerator(X_te, y_test, n_way=n_classes, k_shot=K_SHOT, q_query=Q_QUERY)
    val_task_gen = None if X_va is None else BalancedTaskGenerator(X_va, y_val, n_way=n_classes, k_shot=K_SHOT, q_query=Q_QUERY)

    # Build model
    input_dim = X_tr.shape[1]
    model = HyperMetaLearner(
        input_dim=input_dim,
        hidden_dim=model_cfg["hidden_dim"],
        feature_dim=model_cfg["feature_dim"],
        context_dim=model_cfg["context_dim"],
        num_classes=n_classes,
        use_metric_learner=model_cfg["use_metric_learner"],
        distance_metric=model_cfg["distance_metric"],
        distance_scale_init=model_cfg["distance_scale_init"],
        dropout_p1=model_cfg["dropout_p1"],
        dropout_p2=model_cfg["dropout_p2"],
        use_class_lstm=model_cfg.get("use_class_lstm", True),
        use_feature_net=model_cfg.get("use_feature_net", True),
        prototype_agg=model_cfg.get("prototype_agg", None),
        feature_net_n_hidden=model_cfg.get("feature_net_n_hidden", 2),
    ).to(device)

    # Train
    best_state_dict, _ = train_model(
        model,
        train_task_gen,
        val_task_gen,
        n_classes,
        num_epochs=train_cfg["num_epochs"],
        meta_batch_size=train_cfg["meta_batch_size"],
        patience=train_cfg["patience"],
        lr=train_cfg["lr"],
        weight_decay=train_cfg["weight_decay"],
        scheduler_patience=train_cfg["scheduler_patience"],
        scheduler_factor=train_cfg["scheduler_factor"],
        explicit_l2=train_cfg["explicit_l2"],
    )

    # Load best model and test
    model.load_state_dict(best_state_dict)
    test_acc, all_preds, all_labels, all_probs = test_model(
        model, test_task_gen, n_classes, meta_batch_size=train_cfg["meta_batch_size"]
    )

    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    all_probs = np.array(all_probs)
    macro_auc, _ = calculate_auc(all_probs, all_labels, n_classes)

    # Clinical-style subject metrics: use all test utterances once (no episodic sampling)
    prototypes = compute_class_prototypes_from_train(model, X_tr, y_tr, n_classes)
    test_probs_full = predict_probs_from_prototypes(model, prototypes, X_te)

    do_subject_debug = bool(DEBUG_SUBJECT and (not DEBUG_SUBJECT_ONLY_FIRST_SEED or seed == INIT_SEEDS[0]))
    if do_subject_debug:
        subject_acc, subject_f1, subject_auc, _details = subject_metrics_from_all_test_utterances(
            subj_te, y_test, test_probs_full, n_classes, return_details=True
        )

        tqdm.write('\n' + '=' * 70)
        tqdm.write('SUBJECT-LEVEL EVALUATION (Notebook 3)')
        tqdm.write('=' * 70)
        tqdm.write('This subject score uses ALL test utterances once (not episodic sampling).')
        tqdm.write(f'prototypes shape: {tuple(prototypes.shape)} (one prototype per class)')
        tqdm.write(f'test_probs_full shape: {tuple(test_probs_full.shape)} (one prob vector per test utterance)')

        correct = int(_details['correct'])
        n_subj = int(_details['n_subjects'])
        tqdm.write(f'subject_acc = {correct}/{n_subj} = {subject_acc:.4f}')

        n_utts = np.array(_details['n_utts'], dtype=int)
        tqdm.write(f'test utterances per subject: min={int(n_utts.min())}, median={float(np.median(n_utts)):.1f}, max={int(n_utts.max())}')
        mapping = {i: str(v) for i, v in enumerate(le.classes_)}
        tqdm.write(f"label mapping (index -> name): {mapping}")

        # Show first few subjects so you can see exactly how each subject prediction is formed
        max_show = int(min(DEBUG_SUBJECT_MAX, n_subj))
        tqdm.write(f'\nFirst {max_show} subjects (avg_prob over that subject\'s test utterances):')
        for i in range(max_show):
            sid = _details['subjects'][i]
            yt = int(_details['y_true_sub'][i])
            yp = int(_details['y_pred_sub'][i])
            avg_p = np.asarray(_details['y_prob_sub'][i], dtype=float)
            tqdm.write(f'  {i+1:02d}) subject={sid}  n_utts={int(n_utts[i])}  true={le.classes_[yt]}  pred={le.classes_[yp]}  avg_prob={np.round(avg_p, 4)}')
    else:
        subject_acc, subject_f1, subject_auc = subject_metrics_from_all_test_utterances(
            subj_te, y_test, test_probs_full, n_classes
        )


    # Save per-subject outputs (one row per subject per init seed) for later statistical analysis
    try:
        if not do_subject_debug:
            _, _, _, _details = subject_metrics_from_all_test_utterances(
                subj_te, y_test, test_probs_full, n_classes, return_details=True
            )

        DETAILS_CSV.parent.mkdir(parents=True, exist_ok=True)
        file_exists = DETAILS_CSV.exists()
        fieldnames = [
            'model', 'config', 'smote', 'merge', 'split_seed', 'init_seed',
            'subject_id', 'n_utts', 'y_true', 'y_pred', 'true_name', 'pred_name',
        ] + [f'prob_{i}' for i in range(n_classes)]

        with open(DETAILS_CSV, 'a', newline='') as f:
            w = csv.DictWriter(f, fieldnames=fieldnames)
            if not file_exists:
                w.writeheader()

            label_names = [str(v) for v in le.classes_]
            for sid, n_u, yt, yp, p in zip(
                _details['subjects'],
                _details['n_utts'],
                _details['y_true_sub'],
                _details['y_pred_sub'],
                _details['y_prob_sub'],
            ):
                yt_i = int(yt)
                yp_i = int(yp)
                row = {
                    'model': model_name,
                    'config': config_name,
                    'smote': bool(do_smote),
                    'merge': bool(merge_train_val),
                    'split_seed': int(SPLIT_SEED),
                    'init_seed': int(seed),
                    'subject_id': str(sid),
                    'n_utts': int(n_u),
                    'y_true': yt_i,
                    'y_pred': yp_i,
                    'true_name': label_names[yt_i] if 0 <= yt_i < len(label_names) else str(yt_i),
                    'pred_name': label_names[yp_i] if 0 <= yp_i < len(label_names) else str(yp_i),
                }
                p = np.asarray(p, dtype=float).ravel()
                for i in range(n_classes):
                    row[f'prob_{i}'] = float(p[i])
                w.writerow(row)
    except Exception as e:
        # Never break training/eval because logging failed
        if DEBUG_SUBJECT:
            tqdm.write(f'[WARN] Failed to write subject details CSV ({DETAILS_CSV}): {e}')
    return {
        "test_acc": test_acc,
        "macro_f1": macro_f1,
        "macro_auc": macro_auc,
        "subject_acc": subject_acc,
        "subject_f1": subject_f1,
        "subject_auc": subject_auc,
    }

def run_combination(
    model_name: str,
    config_name: str,
    config_path: Path,
    do_smote: bool,
    merge_train_val: bool,
    model_cfg: dict,
    train_cfg: dict,
) -> Tuple[Dict[str, float], Dict[str, float]]:
    """Run 30 seeds for given (model, config). Returns mean and std of metrics."""
    model_dir = EMB_ROOT / model_name
    X_train, y_train, X_val, y_val, X_test, y_test, subject_ids_val, subject_ids_test, le, n_classes = load_data(
        model_name, config_path, model_dir
    )

    # Save split + label metadata once per (model, config, smote, merge)
    try:
        subj_te_meta = np.asarray(subject_ids_test, dtype=object)
        if merge_train_val and subject_ids_val is not None:
            subj_te_meta = np.concatenate([subj_te_meta, np.asarray(subject_ids_val, dtype=object)], axis=0)
        test_subjects = sorted(np.unique(subj_te_meta.astype(str)).tolist())
        meta = {
            "model": model_name,
            "config": config_name,
            "smote": bool(do_smote),
            "merge": bool(merge_train_val),
            "split_seed": int(SPLIT_SEED),
            "n_classes": int(n_classes),
            "label_names": [str(v) for v in le.classes_],
            "n_test_subjects": int(len(test_subjects)),
            "test_subjects": test_subjects,
        }
        with open(DETAILS_META_JSON, "a", encoding="utf-8") as f:
            f.write(json.dumps(meta, ensure_ascii=False) + "\n")
    except Exception as e:
        if DEBUG_SUBJECT:
            tqdm.write(f"[WARN] Failed to write meta JSONL ({DETAILS_META_JSON}): {e}")

    # Store results per seed
    all_metrics = {"test_acc": [], "macro_f1": [], "macro_auc": [], "subject_acc": [], "subject_f1": [], "subject_auc": []}

    for seed in tqdm(INIT_SEEDS, desc=f"Seeds for {model_name}/{config_name}"):
        metrics = run_single_seed(
            seed, X_train, y_train, X_val, y_val, X_test, y_test,
            subject_ids_val, subject_ids_test,
            n_classes, le, do_smote, merge_train_val, model_cfg, train_cfg,
            model_name=model_name,
            config_name=config_name,
        )
        for k, v in metrics.items():
            all_metrics[k].append(v)

    # Compute mean and std
    mean_dict = {f"{k}_mean": np.mean(vals) for k, vals in all_metrics.items()}
    std_dict = {f"{k}_std": np.std(vals) for k, vals in all_metrics.items()}
    return mean_dict, std_dict

# ====================== SUMMARY CSV HANDLING ======================
def combination_done(model_name: str, config_name: str, do_smote: bool, merge_train_val: bool) -> bool:
    """Check if a row with these exact settings already exists in summary CSV."""
    if not SUMMARY_CSV.exists():
        return False
    with open(SUMMARY_CSV, "r") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if (row["model"] == model_name and
                row["config"] == config_name and
                row["smote"] == str(do_smote) and
                row["merge"] == str(merge_train_val)):
                return True
    return False

def update_summary(
    model_name: str,
    config_name: str,
    do_smote: bool,
    merge_train_val: bool,
    mean_dict: Dict[str, float],
    std_dict: Dict[str, float]
):
    """Append one row to summary CSV."""
    fieldnames = ["model", "config", "smote", "merge",
                  "test_acc_mean", "test_acc_std",
                  "macro_f1_mean", "macro_f1_std",
                  "macro_auc_mean", "macro_auc_std",
                  "subject_acc_mean", "subject_acc_std",
                  "subject_f1_mean", "subject_f1_std",
                  "subject_auc_mean", "subject_auc_std"]
    row = {
        "model": model_name,
        "config": config_name,
        "smote": str(do_smote),
        "merge": str(merge_train_val),
        **mean_dict,
        **std_dict,
    }
    out_path = SUMMARY_CSV
    try:
        file_exists = out_path.exists()
        with open(out_path, "a", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            if not file_exists:
                writer.writeheader()
            writer.writerow(row)
    except PermissionError as e:
        # Common on Windows/WSL when the CSV is open in Excel. Fall back to an autosave file.
        out_path = RESULTS_DIR / f"{SUMMARY_CSV.stem}_autosave.csv"
        file_exists = out_path.exists()
        with open(out_path, "a", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            if not file_exists:
                writer.writeheader()
            writer.writerow(row)
        tqdm.write(f"[WARN] Summary CSV was locked ({SUMMARY_CSV}). Wrote to: {out_path} ({e})")

# ====================== MAIN SCRIPT ======================

def _pick_best_from_summary(path: Path, *, metric: str = "test_acc_mean") -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing summary CSV: {path}")
    with open(path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        best_row = None
        best_val = None
        for row in reader:
            try:
                v = float(row[metric])
            except Exception:
                continue
            if best_val is None or v > best_val:
                best_val = v
                best_row = row
    if best_row is None:
        raise ValueError(f"No valid rows found in summary CSV for metric '{metric}': {path}")
    return best_row


def _find_config_path(model_dir: Path, model_name: str, config_name: str) -> Path:
    cfgs = discover_configs(model_dir, model_name)
    for name, p in cfgs:
        if name == config_name:
            return p
    available = [n for n, _ in cfgs]
    raise FileNotFoundError(f"Config '{config_name}' not found in {model_dir}. Available: {available}")


def _ablation_already_done(out_csv: Path, *, base_model: str, base_config: str, variant: str, smote: bool, merge: bool, k_shot: int, q_query: int) -> bool:
    if not out_csv.exists():
        return False
    try:
        with open(out_csv, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for r in reader:
                if (r.get("base_model") == base_model and
                    r.get("base_config") == base_config and
                    r.get("variant") == variant and
                    r.get("smote") == str(bool(smote)) and
                    r.get("merge") == str(bool(merge)) and
                    int(float(r.get("k_shot", -1))) == int(k_shot) and
                    int(float(r.get("q_query", -1))) == int(q_query)):
                    return True
    except Exception:
        return False
    return False


def _append_ablation_summary(row: dict, out_csv: Path):
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    file_exists = out_csv.exists()
    fieldnames = list(row.keys())
    with open(out_csv, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            w.writeheader()
        w.writerow(row)


def main():
    print("=== ABLATION RUNNER (best model+config only) ===")

    # Choose the best-performing (model, config) from the main summary CSV.
    # This matches your request: run ablations only on the best setting.
    BEST_BY = "test_acc_mean"  # change if you want: macro_f1_mean / macro_auc_mean / subject_f1_mean ...
    base_summary = RESULTS_DIR / "experiment_summary_subject_fulltest.csv"
    best = _pick_best_from_summary(base_summary, metric=BEST_BY)

    best_model = best["model"]
    best_config = best["config"]
    do_smote = (best["smote"] == "True")
    merge_train_val = (best["merge"] == "True")

    print(f"Picked BEST_BY={BEST_BY}: model={best_model} config={best_config} smote={do_smote} merge={merge_train_val}")

    model_dir = EMB_ROOT / best_model
    config_path = _find_config_path(model_dir, best_model, best_config)

    # Clean, fixed output folder (no run_ timestamps)
    ablation_dir = RESULTS_DIR / "exp9_ablation_bestconfig"
    ablation_dir.mkdir(parents=True, exist_ok=True)

    global SUMMARY_CSV, DETAILS_CSV, DETAILS_META_JSON
    SUMMARY_CSV = ablation_dir / "ablation_summary_bestconfig.csv"
    DETAILS_CSV = ablation_dir / "ablation_subject_level_details.csv"
    DETAILS_META_JSON = ablation_dir / "ablation_subject_level_meta.jsonl"

    print(f"Ablation results: {ablation_dir}")
    print(f"K_SHOT={K_SHOT} | Q_QUERY={Q_QUERY} | split_seed={SPLIT_SEED} | n_init_seeds={len(INIT_SEEDS)}")

    # Baseline + 4 ablations (as you requested)
    variants = {
        "baseline": {},
        "no_class_lstm": {"use_class_lstm": False},
        "cnn_instead_of_lstm": {"prototype_agg": "cnn"},
        "no_metric_learner": {"use_metric_learner": False},
        "euclidean_distance": {"distance_metric": "euclidean"},
        "no_feature_net": {"use_feature_net": False},
        "feature_net_shallow": {"use_feature_net": True, "feature_net_n_hidden": 1},
        "feature_net_deeper": {"use_feature_net": True, "feature_net_n_hidden": 3},
    }

    # If you want to run fewer, edit this list:
    run_variants = [
        "baseline",
        "no_class_lstm",
        "cnn_instead_of_lstm",
        "no_metric_learner",
        "euclidean_distance",
        "no_feature_net",
        "feature_net_shallow",
        "feature_net_deeper",
    ]

    for variant in run_variants:
        overrides = variants[variant]
        cfg = dict(MODEL_CFG)
        cfg.update(overrides)

        tag = f"{best_config}__{variant}"
        print()
        print("-" * 90)
        print(f"RUN: model={best_model} | config={best_config} | variant={variant}")
        print(f"X: {config_path}")
        print(f"Overrides: {overrides}")

        if _ablation_already_done(SUMMARY_CSV, base_model=best_model, base_config=best_config, variant=variant, smote=do_smote, merge=merge_train_val, k_shot=K_SHOT, q_query=Q_QUERY):
            print("Skipping (already in ablation summary).")
            continue

        mean_dict, std_dict = run_combination(
            best_model,
            tag,
            config_path,
            do_smote,
            merge_train_val,
            cfg,
            TRAIN_CFG,
        )

        out_row = {
            "best_by": BEST_BY,
            "base_model": best_model,
            "base_config": best_config,
            "variant": variant,
            "smote": bool(do_smote),
            "merge": bool(merge_train_val),
            "split_seed": int(SPLIT_SEED),
            "k_shot": int(K_SHOT),
            "q_query": int(Q_QUERY),
            **mean_dict,
            **std_dict,
        }
        _append_ablation_summary(out_row, SUMMARY_CSV)
        print(f"Appended summary row -> {SUMMARY_CSV}")

    print()
    print("DONE")
    print(f"Summary CSV: {SUMMARY_CSV}")
    print(f"Subject details CSV: {DETAILS_CSV}")
    print(f"Meta JSONL: {DETAILS_META_JSON}")


if __name__ == "__main__":
    main()


=== ABLATION RUNNER (best model+config only) ===
Picked BEST_BY=test_acc_mean: model=wav2vec config=stability smote=True merge=True
Ablation results: /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/Results/AL_MODELS/exp9_ablation_bestconfig
K_SHOT=5 | Q_QUERY=15 | split_seed=42 | n_init_seeds=30

------------------------------------------------------------------------------------------
RUN: model=wav2vec | config=stability | variant=baseline
X: /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/embeddings/wav2vec/xwav2vec2_stability.npy
Overrides: {}
Skipping (already in ablation summary).

------------------------------------------------------------------------------------------
RUN: model=wav2vec | config=stability | variant=no_class_lstm
X: /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/embeddings/wav2vec/xwav2vec2_stability.npy
Overrides: {'use_class_lstm': False}
Skipping (already in ablation summary).

------------------------------------------------------------------------------------------
RUN:

Seeds for wav2vec/stability__cnn_instead_of_lstm:   3%|▎         | 1/30 [00:20<09:50, 20.36s/it]


SUBJECT-LEVEL EVALUATION (Notebook 3)
This subject score uses ALL test utterances once (not episodic sampling).
prototypes shape: (3, 256) (one prototype per class)
test_probs_full shape: (1433, 3) (one prob vector per test utterance)
subject_acc = 12/13 = 0.9231
test utterances per subject: min=100, median=112.0, max=118
label mapping (index -> name): {0: '0', 1: '1', 2: '2'}

First 10 subjects (avg_prob over that subject's test utterances):
  01) subject=ALSwDysarthria/F1  n_utts=115  true=1  pred=1  avg_prob=[0.0574 0.9091 0.0335]
  02) subject=ALSwDysarthria/F10  n_utts=107  true=1  pred=1  avg_prob=[0.0953 0.5516 0.3531]
  03) subject=ALSwDysarthria/F6  n_utts=106  true=1  pred=1  avg_prob=[0.0173 0.6613 0.3214]
  04) subject=ALSwDysarthria/M3  n_utts=109  true=1  pred=1  avg_prob=[0.0096 0.9141 0.0763]
  05) subject=ALSwDysarthria/M4  n_utts=100  true=1  pred=2  avg_prob=[2.000e-04 3.665e-01 6.333e-01]
  06) subject=ALSwDysarthria/M6  n_utts=118  true=1  pred=1  avg_prob=[0.008 

Seeds for wav2vec/stability__cnn_instead_of_lstm: 100%|██████████| 30/30 [08:51<00:00, 17.72s/it]


Appended summary row -> /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/Results/AL_MODELS/exp9_ablation_bestconfig/ablation_summary_bestconfig.csv

------------------------------------------------------------------------------------------
RUN: model=wav2vec | config=stability | variant=no_metric_learner
X: /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/embeddings/wav2vec/xwav2vec2_stability.npy
Overrides: {'use_metric_learner': False}
Skipping (already in ablation summary).

------------------------------------------------------------------------------------------
RUN: model=wav2vec | config=stability | variant=euclidean_distance
X: /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/embeddings/wav2vec/xwav2vec2_stability.npy
Overrides: {'distance_metric': 'euclidean'}
Skipping (already in ablation summary).

------------------------------------------------------------------------------------------
RUN: model=wav2vec | config=stability | variant=no_feature_net
X: /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Me

...

Appended summary row -> /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/Results/AL_MODELS/exp9_ablation_bestconfig/ablation_summary_bestconfig.csv

------------------------------------------------------------------------------------------
RUN: model=wav2vec | config=stability | variant=feature_net_deeper
X: /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/embeddings/wav2vec/xwav2vec2_stability.npy
Overrides: {'use_feature_net': True, 'feature_net_n_hidden': 3}


Seeds for wav2vec/stability__feature_net_deeper:   3%|▎         | 1/30 [00:19<09:25, 19.50s/it]


SUBJECT-LEVEL EVALUATION (Notebook 3)
This subject score uses ALL test utterances once (not episodic sampling).
prototypes shape: (3, 256) (one prototype per class)
test_probs_full shape: (1433, 3) (one prob vector per test utterance)
subject_acc = 12/13 = 0.9231
test utterances per subject: min=100, median=112.0, max=118
label mapping (index -> name): {0: '0', 1: '1', 2: '2'}

First 10 subjects (avg_prob over that subject's test utterances):
  01) subject=ALSwDysarthria/F1  n_utts=115  true=1  pred=1  avg_prob=[0.0549 0.9031 0.042 ]
  02) subject=ALSwDysarthria/F10  n_utts=107  true=1  pred=1  avg_prob=[0.0877 0.5758 0.3365]
  03) subject=ALSwDysarthria/F6  n_utts=106  true=1  pred=1  avg_prob=[0.0261 0.6815 0.2924]
  04) subject=ALSwDysarthria/M3  n_utts=109  true=1  pred=1  avg_prob=[0.0041 0.9317 0.0643]
  05) subject=ALSwDysarthria/M4  n_utts=100  true=1  pred=2  avg_prob=[2.000e-04 3.802e-01 6.196e-01]
  06) subject=ALSwDysarthria/M6  n_utts=118  true=1  pred=1  avg_prob=[0.0076

Seeds for wav2vec/stability__feature_net_deeper: 100%|██████████| 30/30 [09:55<00:00, 19.83s/it]

Appended summary row -> /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/Results/AL_MODELS/exp9_ablation_bestconfig/ablation_summary_bestconfig.csv

DONE
Summary CSV: /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/Results/AL_MODELS/exp9_ablation_bestconfig/ablation_summary_bestconfig.csv
Subject details CSV: /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/Results/AL_MODELS/exp9_ablation_bestconfig/ablation_subject_level_details.csv
Meta JSONL: /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/Results/AL_MODELS/exp9_ablation_bestconfig/ablation_subject_level_meta.jsonl
